# 🥇 Gold — Analytics Avançados + Otimização

## Features implementadas

| # | Feature | Descrição |
|---|---------|----------|
| 1 | **MERGE incremental** | Todas as tabelas Gold sem overwrite |
| 2 | **RFM completo** | Quintis de Recency/Frequency/Monetary |
| 3 | **Segmentação** | Champions, Loyal, At Risk, Lost, etc. |
| 4 | **Churn Risk** | Clientes inativos por nível de risco |
| 5 | **Product Performance** | Receita, unidades, rank por categoria |
| 6 | **Cohort Analysis** | Retenção por coorte de aquisição |
| 7 | **YoY / MoM Growth** | Crescimento com `LAG()` |
| 8 | **Percentil / Histograma** | Distribuição de ticket com `percentile_approx` |
| 9 | **Pivot Table** | Receita por categoria × país |
| 10 | **Market Basket (FP-Growth)** | Produtos frequentemente comprados juntos |
| 11 | `OPTIMIZE + Z-ORDER + VACUUM` | Manutenção física de todas as tabelas |
| 12 | `ANALYZE TABLE` | Estatísticas para o otimizador (CBO) |
| 13 | **Cache** | Tabelas Gold quentes em memória |

## 0. Configuração

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from functools import reduce

CATALOG = 'workspace'
SCHEMA  = 'medallion_demo'

spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')

silver = spark.table('silver_sales')
print(f'✅ Silver carregada: {silver.count():,} linhas')

## 1. DDL das tabelas Gold + MERGE incremental

Todas as tabelas usam `MERGE` em vez de `overwrite`:
atualiza grupos existentes, insere grupos novos — sem reprocessar tudo.

In [ ]:
gold_ddls = {
    'gold_revenue_by_category': '''
        CREATE TABLE IF NOT EXISTS gold_revenue_by_category (
            category STRING, country STRING, num_transactions LONG,
            revenue DOUBLE, avg_ticket DOUBLE, units_sold LONG
        ) USING DELTA TBLPROPERTIES ("delta.autoOptimize.optimizeWrite" = "true")
    ''',
    'gold_monthly_revenue': '''
        CREATE TABLE IF NOT EXISTS gold_monthly_revenue (
            year_month STRING, revenue DOUBLE, num_transactions LONG, active_customers LONG
        ) USING DELTA TBLPROPERTIES ("delta.autoOptimize.optimizeWrite" = "true")
    ''',
    'gold_customer_value': '''
        CREATE TABLE IF NOT EXISTS gold_customer_value (
            customer_id INT, frequency LONG, monetary DOUBLE, last_purchase DATE
        ) USING DELTA TBLPROPERTIES ("delta.autoOptimize.optimizeWrite" = "true")
    ''',
    'gold_rfm': '''
        CREATE TABLE IF NOT EXISTS gold_rfm (
            customer_id INT, recency_days INT, frequency LONG, monetary DOUBLE,
            r_score INT, f_score INT, m_score INT, rfm_score INT,
            rfm_cell STRING, segment STRING, last_purchase DATE
        ) USING DELTA TBLPROPERTIES ("delta.autoOptimize.optimizeWrite" = "true")
    ''',
    'gold_churn_risk': '''
        CREATE TABLE IF NOT EXISTS gold_churn_risk (
            customer_id INT, last_purchase DATE, days_inactive INT,
            total_orders LONG, total_spent DOUBLE, churn_risk STRING
        ) USING DELTA TBLPROPERTIES ("delta.autoOptimize.optimizeWrite" = "true")
    ''',
    'gold_product_performance': '''
        CREATE TABLE IF NOT EXISTS gold_product_performance (
            product_id INT, category STRING, total_units_sold LONG,
            total_revenue DOUBLE, avg_unit_price DOUBLE,
            num_transactions LONG, distinct_buyers LONG
        ) USING DELTA TBLPROPERTIES ("delta.autoOptimize.optimizeWrite" = "true")
    ''',
    'gold_cohort': '''
        CREATE TABLE IF NOT EXISTS gold_cohort (
            cohort_month STRING, months_since_acquisition INT, active_customers LONG,
            cohort_size LONG, retention_pct DOUBLE
        ) USING DELTA TBLPROPERTIES ("delta.autoOptimize.optimizeWrite" = "true")
    ''',
    'gold_revenue_trend': '''
        CREATE TABLE IF NOT EXISTS gold_revenue_trend (
            year_month STRING, revenue DOUBLE, prev_month_revenue DOUBLE,
            mom_growth_pct DOUBLE, prev_year_revenue DOUBLE, yoy_growth_pct DOUBLE
        ) USING DELTA TBLPROPERTIES ("delta.autoOptimize.optimizeWrite" = "true")
    ''',
    'gold_ticket_distribution': '''
        CREATE TABLE IF NOT EXISTS gold_ticket_distribution (
            category STRING, p25 DOUBLE, p50 DOUBLE, p75 DOUBLE,
            p90 DOUBLE, p95 DOUBLE, p99 DOUBLE, mean DOUBLE, stddev DOUBLE
        ) USING DELTA TBLPROPERTIES ("delta.autoOptimize.optimizeWrite" = "true")
    ''',
    'gold_market_basket': '''
        CREATE TABLE IF NOT EXISTS gold_market_basket (
            antecedent ARRAY<INT>, consequent ARRAY<INT>,
            confidence DOUBLE, lift DOUBLE, support DOUBLE
        ) USING DELTA TBLPROPERTIES ("delta.autoOptimize.optimizeWrite" = "true")
    '''
}

for tbl, ddl in gold_ddls.items():
    spark.sql(ddl)
    print(f'✅ {tbl}')

In [ ]:
def merge_into(target_name, source_df, join_keys):
    join_cond = ' AND '.join([f'tgt.{k} = src.{k}' for k in join_keys])
    (DeltaTable.forName(spark, target_name).alias('tgt')
        .merge(source_df.alias('src'), join_cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    cnt = spark.table(target_name).count()
    print(f'✅ MERGE {target_name}: {cnt:,} linhas')

# Receita por categoria e país
merge_into('gold_revenue_by_category',
    silver.groupBy('category', 'country').agg(
        F.count('*').alias('num_transactions'),
        F.round(F.sum('total_amount'), 2).alias('revenue'),
        F.round(F.avg('total_amount'), 2).alias('avg_ticket'),
        F.sum('quantity').alias('units_sold'),
    ),
    ['category', 'country'])

# Receita mensal
merge_into('gold_monthly_revenue',
    silver.withColumn('year_month', F.date_format('event_date', 'yyyy-MM'))
    .groupBy('year_month').agg(
        F.round(F.sum('total_amount'), 2).alias('revenue'),
        F.count('*').alias('num_transactions'),
        F.countDistinct('customer_id').alias('active_customers'),
    ),
    ['year_month'])

# Valor do cliente
merge_into('gold_customer_value',
    silver.groupBy('customer_id').agg(
        F.count('*').alias('frequency'),
        F.round(F.sum('total_amount'), 2).alias('monetary'),
        F.max('event_date').alias('last_purchase'),
    ),
    ['customer_id'])

# Performance de produto
merge_into('gold_product_performance',
    silver.groupBy('product_id', 'category').agg(
        F.sum('quantity').alias('total_units_sold'),
        F.round(F.sum('total_amount'), 2).alias('total_revenue'),
        F.round(F.avg('unit_price'), 2).alias('avg_unit_price'),
        F.count('*').alias('num_transactions'),
        F.countDistinct('customer_id').alias('distinct_buyers'),
    ),
    ['product_id', 'category'])

## 2. RFM Scoring com quintis + Segmentação

| Score | R (Recency) | F (Frequency) | M (Monetary) |
|-------|-------------|--------------|-------------|
| 5 | Comprou há 0–20% dias | Top 20% mais frequente | Top 20% maior gasto |
| 1 | Comprou há 80–100% dias | Bottom 20% | Bottom 20% |

In [ ]:
today = F.current_date()

rfm_base = (
    silver.groupBy('customer_id')
    .agg(
        F.datediff(today, F.max('event_date')).alias('recency_days'),
        F.count('*').alias('frequency'),
        F.round(F.sum('total_amount'), 2).alias('monetary'),
        F.max('event_date').alias('last_purchase'),
    )
)

# ntile(5): divide em 5 grupos iguais por ordem crescente
# Recency: menor recency_days = mais recente = score MAIOR (por isso invertemos com 6-ntile)
rfm_scored = (
    rfm_base
    .withColumn('r_score', 6 - F.ntile(5).over(Window.orderBy('recency_days')))
    .withColumn('f_score', F.ntile(5).over(Window.orderBy('frequency')))
    .withColumn('m_score', F.ntile(5).over(Window.orderBy('monetary')))
    .withColumn('rfm_score', F.col('r_score') + F.col('f_score') + F.col('m_score'))
    .withColumn('rfm_cell',  F.concat(F.col('r_score').cast('string'), F.col('f_score').cast('string'), F.col('m_score').cast('string')))
)

rfm_final = rfm_scored.withColumn('segment',
    F.when(F.col('rfm_score') >= 13,                                               'Champions')
    .when((F.col('r_score') >= 4) & (F.col('f_score') >= 4),                      'Loyal Customers')
    .when((F.col('r_score') >= 4) & (F.col('rfm_score') >= 9),                    'Potential Loyalists')
    .when((F.col('r_score') == 5) & (F.col('f_score') <= 2),                      'New Customers')
    .when((F.col('r_score') <= 2) & (F.col('f_score') >= 4),                      'At Risk')
    .when((F.col('r_score') <= 2) & (F.col('f_score') <= 2),                      'Lost')
    .otherwise('Needs Attention')
)

merge_into('gold_rfm', rfm_final, ['customer_id'])

print('\n=== Segmentos de clientes ===')
display(
    spark.table('gold_rfm')
    .groupBy('segment')
    .agg(
        F.count('*').alias('clientes'),
        F.round(F.sum('monetary'), 2).alias('receita_total'),
        F.round(F.avg('rfm_score'), 1).alias('score_medio'),
        F.round(F.avg('recency_days'), 0).alias('recencia_media_dias'),
    )
    .orderBy(F.col('score_medio').desc())
)

## 3. Churn Risk Analysis

| Risco | Critério | Ação sugerida |
|-------|----------|---------------|
| 🔴 Alto | > 180 dias | Campanha de reativação urgente |
| 🟡 Médio | 90–180 dias | Desconto de retorno |
| 🟢 Baixo | 60–89 dias | Lembrete por e-mail |

In [ ]:
churn_data = (
    silver.groupBy('customer_id')
    .agg(
        F.max('event_date').alias('last_purchase'),
        F.count('*').alias('total_orders'),
        F.round(F.sum('total_amount'), 2).alias('total_spent'),
    )
    .withColumn('days_inactive', F.datediff(F.current_date(), F.col('last_purchase')))
    .withColumn('churn_risk',
        F.when(F.col('days_inactive') > 180, 'Alto')
        .when(F.col('days_inactive') > 90,   'Medio')
        .otherwise('Baixo')
    )
    .filter(F.col('days_inactive') >= 60)
)

merge_into('gold_churn_risk', churn_data, ['customer_id'])

display(
    spark.table('gold_churn_risk')
    .groupBy('churn_risk')
    .agg(
        F.count('*').alias('clientes'),
        F.round(F.avg('days_inactive'), 1).alias('dias_inativos_medio'),
        F.round(F.sum('total_spent'), 2).alias('receita_em_risco'),
    )
    .orderBy(F.col('dias_inativos_medio').desc())
)

## 4. Cohort Analysis — Retenção por mês de aquisição

**O que é?** Agrupa clientes pelo mês em que fizeram a **primeira compra** (coorte).
Depois rastreia quantos % ainda compram nos meses seguintes.

**Para que serve?** Mede a qualidade da aquisição de clientes:
- Uma coorte com alta retenção em M+3 indica produto/mercado adequados
- Queda brusca em M+1 indica problema de experiência inicial

In [ ]:
# Mês de primeira compra por cliente (coorte)
cohort_base = (
    silver.groupBy('customer_id')
    .agg(F.min('event_date').alias('first_purchase_date'))
    .withColumn('cohort_month', F.date_format('first_purchase_date', 'yyyy-MM'))
)

# Tamanho de cada coorte (quantos clientes adquiridos naquele mês)
cohort_size = (
    cohort_base.groupBy('cohort_month')
    .agg(F.countDistinct('customer_id').alias('cohort_size'))
)

# Join das compras com a coorte do cliente
cohort_activity = (
    silver
    .join(cohort_base, 'customer_id', 'inner')
    .withColumn(
        'months_since_acquisition',
        F.months_between(F.col('event_date'), F.col('first_purchase_date')).cast('int')
    )
    .filter(F.col('months_since_acquisition') >= 0)  # apenas compras após aquisição
)

# Clientes ativos por coorte e mês desde aquisição
retention_raw = (
    cohort_activity
    .groupBy('cohort_month', 'months_since_acquisition')
    .agg(F.countDistinct('customer_id').alias('active_customers'))
)

# Junta com tamanho da coorte e calcula % de retenção
cohort_final = (
    retention_raw
    .join(cohort_size, 'cohort_month', 'left')
    .withColumn('retention_pct', F.round(F.col('active_customers') / F.col('cohort_size') * 100, 1))
    .orderBy('cohort_month', 'months_since_acquisition')
)

merge_into('gold_cohort', cohort_final, ['cohort_month', 'months_since_acquisition'])

print('=== Matriz de retenção por coorte (M+0 a M+3) ===')
display(
    spark.table('gold_cohort')
    .filter(F.col('months_since_acquisition') <= 3)
    .groupBy('cohort_month')
    .pivot('months_since_acquisition', [0, 1, 2, 3])
    .agg(F.first('retention_pct'))
    .orderBy('cohort_month')
)

## 5. YoY / MoM Growth — Crescimento com LAG()

- **MoM (Month-over-Month)**: crescimento vs mês anterior (`LAG(revenue, 1)`)
- **YoY (Year-over-Year)**: crescimento vs mesmo mês do ano anterior (`LAG(revenue, 12)`)

In [ ]:
w_time = Window.orderBy('year_month')

monthly = spark.table('gold_monthly_revenue')

trend = (
    monthly
    .withColumn('prev_month_revenue', F.lag('revenue', 1).over(w_time))
    .withColumn('mom_growth_pct',
        F.round(
            (F.col('revenue') - F.col('prev_month_revenue')) / F.col('prev_month_revenue') * 100,
            2
        )
    )
    .withColumn('prev_year_revenue', F.lag('revenue', 12).over(w_time))
    .withColumn('yoy_growth_pct',
        F.round(
            (F.col('revenue') - F.col('prev_year_revenue')) / F.col('prev_year_revenue') * 100,
            2
        )
    )
    .select('year_month', 'revenue', 'prev_month_revenue', 'mom_growth_pct',
            'prev_year_revenue', 'yoy_growth_pct')
)

merge_into('gold_revenue_trend', trend, ['year_month'])

print('=== Tendência de receita com crescimento MoM e YoY ===')
display(
    spark.table('gold_revenue_trend')
    .orderBy('year_month')
    .select('year_month', 'revenue', 'mom_growth_pct', 'yoy_growth_pct')
)

## 6. Percentil / Histograma — Distribuição de ticket

`percentile_approx` usa o algoritmo de Greenwald-Khanna para calcular percentis
em datasets grandes sem mover todos os dados para o driver.

Útil para detectar: outliers, distribuição por cauda, pricing adequado.

In [ ]:
# Distribuição por categoria
ticket_dist = (
    silver.groupBy('category')
    .agg(
        F.round(F.percentile_approx('total_amount', 0.25), 2).alias('p25'),
        F.round(F.percentile_approx('total_amount', 0.50), 2).alias('p50'),
        F.round(F.percentile_approx('total_amount', 0.75), 2).alias('p75'),
        F.round(F.percentile_approx('total_amount', 0.90), 2).alias('p90'),
        F.round(F.percentile_approx('total_amount', 0.95), 2).alias('p95'),
        F.round(F.percentile_approx('total_amount', 0.99), 2).alias('p99'),
        F.round(F.avg('total_amount'), 2).alias('mean'),
        F.round(F.stddev('total_amount'), 2).alias('stddev'),
    )
    .orderBy('category')
)

merge_into('gold_ticket_distribution', ticket_dist, ['category'])

print('=== Distribuição de ticket por categoria ===')
display(spark.table('gold_ticket_distribution'))

# Histograma global (10 buckets) de total_amount
print('\n=== Histograma de ticket (10 buckets) ===')
histogram = (
    silver
    .select(F.histogram_numeric('total_amount', F.lit(10)).alias('hist'))
    .select(F.explode('hist').alias('bucket'))
    .select(
        F.round(F.col('bucket.x'), 2).alias('center'),
        F.col('bucket.y').cast('long').alias('count'),
    )
    .orderBy('center')
)
display(histogram)

## 7. Pivot Table — Receita por categoria × país

`pivot()` transforma linhas em colunas — cria uma tabela matricial
(linhas = categorias, colunas = países) ideal para análise cross-dimensional.

In [ ]:
paises = ['BR', 'US', 'AR', 'CL', 'MX', 'PT']

# Pivot: categoria nas linhas, países nas colunas, receita nos valores
pivot_revenue = (
    silver
    .groupBy('category')
    .pivot('country', paises)
    .agg(F.round(F.sum('total_amount'), 2))
    .orderBy('category')
)

print('=== Receita por categoria × país (Pivot Table) ===')
display(pivot_revenue)

# Pivot de método de pagamento por país
print('\n=== Transações por método de pagamento × país ===')
pivot_payment = (
    silver
    .groupBy('payment_method')
    .pivot('country', paises)
    .agg(F.count('*'))
    .orderBy('payment_method')
)
display(pivot_payment)

## 8. Market Basket Analysis (FP-Growth)

**FP-Growth** (Frequent Pattern Growth) encontra itens frequentemente comprados juntos.

**Métricas-chave:**
- `support` = % de pedidos que contêm o conjunto de produtos
- `confidence` = P(consequent | antecedent) — probabilidade condicional
- `lift` = confidence / P(consequent) — quanto o antecedente aumenta a probabilidade

In [ ]:
from pyspark.ml.fpm import FPGrowth

# Agrupa produtos por pedido (cliente + dia = "pedido")
baskets = (
    silver
    .groupBy('customer_id', 'event_date')
    .agg(F.collect_set('product_id').alias('products'))
    .filter(F.size('products') >= 2)  # apenas pedidos com 2+ produtos
)

basket_count = baskets.count()
print(f'Pedidos com 2+ produtos: {basket_count:,}')

if basket_count > 100:
    # Treina o modelo FP-Growth
    fp = FPGrowth(
        itemsCol='products',
        minSupport=0.005,     # 0.5% dos pedidos devem conter o itemset
        minConfidence=0.1,    # 10% de confiança mínima
    )
    model = fp.fit(baskets)

    # Regras de associação
    rules = (
        model.associationRules
        .withColumn('lift', F.round('lift', 3))
        .withColumn('confidence', F.round('confidence', 3))
        .withColumn('support', F.round('support', 4))
        .orderBy(F.col('lift').desc())
    )

    # Salva na tabela gold
    rules.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold_market_basket')

    print(f'✅ FP-Growth concluído: {rules.count()} regras de associação')
    print('\n=== Top 10 regras (maior lift) ===')
    display(spark.table('gold_market_basket').orderBy(F.col('lift').desc()).limit(10))

    # Itemsets frequentes (sem direção)
    print('\n=== Top 10 itemsets mais frequentes ===')
    display(model.freqItemsets.orderBy(F.col('freq').desc()).limit(10))
else:
    print('ℹ️  Poucos pedidos multi-produto para FP-Growth. Ajuste minSupport.')

## 9. OPTIMIZE + Z-ORDER em todas as tabelas Gold

In [ ]:
optimize_config = [
    ('gold_revenue_by_category',  'category, country'),
    ('gold_monthly_revenue',      'year_month'),
    ('gold_customer_value',       'customer_id'),
    ('gold_rfm',                  'segment, rfm_score'),
    ('gold_churn_risk',           'churn_risk, days_inactive'),
    ('gold_product_performance',  'category, total_revenue'),
    ('gold_cohort',               'cohort_month, months_since_acquisition'),
    ('gold_revenue_trend',        'year_month'),
    ('gold_ticket_distribution',  'category'),
]

for tbl, zorder_cols in optimize_config:
    print(f'⏳ {tbl}...')
    spark.sql(f'OPTIMIZE {tbl} ZORDER BY ({zorder_cols})')
    print(f'✅ {tbl} → Z-ORDER BY ({zorder_cols})')

## 10. VACUUM

In [ ]:
all_gold_tables = [
    'gold_revenue_by_category', 'gold_monthly_revenue', 'gold_customer_value',
    'gold_rfm', 'gold_churn_risk', 'gold_product_performance',
    'gold_cohort', 'gold_revenue_trend', 'gold_ticket_distribution', 'gold_market_basket',
]

for tbl in all_gold_tables:
    try:
        preview = spark.sql(f'VACUUM {tbl} RETAIN 168 HOURS DRY RUN')
        n = preview.count()
        if n > 0:
            spark.sql(f'VACUUM {tbl} RETAIN 168 HOURS')
            print(f'✅ VACUUM {tbl}: {n} arquivo(s) removido(s)')
        else:
            print(f'ℹ️  {tbl}: nenhum arquivo para remover')
    except Exception as e:
        print(f'⚠️  {tbl}: {str(e)[:60]}')

## 11. ANALYZE TABLE — Estatísticas para o CBO

Alimenta o **Cost-Based Optimizer** com min/max/ndv de cada coluna.
O Spark usa essas estatísticas para escolher broadcast vs sort-merge join
e ordenar os joins de forma ótima.

In [ ]:
analyze_config = [
    ('gold_revenue_by_category', 'category, country, revenue'),
    ('gold_monthly_revenue',     'year_month, revenue, active_customers'),
    ('gold_rfm',                 'segment, rfm_score, monetary, customer_id'),
    ('gold_churn_risk',          'churn_risk, days_inactive, total_spent'),
    ('gold_product_performance', 'category, product_id, total_revenue'),
    ('gold_cohort',              'cohort_month, months_since_acquisition, retention_pct'),
    ('gold_revenue_trend',       'year_month, mom_growth_pct, yoy_growth_pct'),
]

for tbl, cols in analyze_config:
    spark.sql(f'ANALYZE TABLE {tbl} COMPUTE STATISTICS FOR COLUMNS {cols}')
    print(f'✅ Estatísticas: {tbl} ({cols})')

## 12. Cache de tabelas Gold quentes

Tabelas acessadas com frequência por dashboards / notebooks analíticos
ficam em memória → resposta imediata nas primeiras consultas.

In [ ]:
import time

hot_tables = [
    'gold_rfm',
    'gold_revenue_by_category',
    'gold_monthly_revenue',
]

for tbl in hot_tables:
    spark.catalog.cacheTable(tbl)
    print(f'✅ {tbl} cacheada: {spark.catalog.isCached(tbl)}')

# Mede ganho de performance
print('\n=== Benchmark: cache vs sem cache ===')

spark.catalog.uncacheTable('gold_rfm')
t0 = time.time()
spark.table('gold_rfm').filter('segment = "Champions"').count()
t_sem_cache = time.time() - t0

spark.catalog.cacheTable('gold_rfm')
spark.table('gold_rfm').count()  # força materialização do cache
t0 = time.time()
spark.table('gold_rfm').filter('segment = "Champions"').count()
t_com_cache = time.time() - t0

print(f'   Sem cache : {t_sem_cache:.3f}s')
print(f'   Com cache : {t_com_cache:.3f}s')
speedup = t_sem_cache / max(t_com_cache, 0.001)
print(f'   Speedup   : {speedup:.1f}×')

## 13. Resumo final — todas as tabelas

In [ ]:
all_tables = [
    ('Bronze',  'bronze_sales'),
    ('Silver',  'silver_sales'),
    ('Silver',  'dim_customer_scd1'),
    ('Silver',  'dim_customer_scd2'),
    ('Gold',    'gold_revenue_by_category'),
    ('Gold',    'gold_monthly_revenue'),
    ('Gold',    'gold_customer_value'),
    ('Gold',    'gold_rfm'),
    ('Gold',    'gold_churn_risk'),
    ('Gold',    'gold_product_performance'),
    ('Gold',    'gold_cohort'),
    ('Gold',    'gold_revenue_trend'),
    ('Gold',    'gold_ticket_distribution'),
    ('Gold',    'gold_market_basket'),
]

print(f'{"Camada":<10} {"Tabela":<35} {"Linhas":>12}')
print('=' * 60)
layer_totals = {}
for layer, tbl in all_tables:
    try:
        cnt = spark.table(tbl).count()
        print(f'{layer:<10} {tbl:<35} {cnt:>12,}')
        layer_totals[layer] = layer_totals.get(layer, 0) + cnt
    except:
        print(f'{layer:<10} {tbl:<35} {"N/A":>12}')
print('=' * 60)
for layer, total in layer_totals.items():
    print(f'{layer:<10} {"TOTAL":<35} {total:>12,}')

print('\n=== DESCRIBE DETAIL — tamanho físico das tabelas Gold ===')
details = [spark.sql(f'DESCRIBE DETAIL {tbl}').select('name', 'numFiles', 'sizeInBytes')
           for _, tbl in all_tables if tbl.startswith('gold_')]
display(reduce(lambda a, b: a.union(b), details))